In [ ]:
!pip install -q openai

from google.colab import userdata
from openai import OpenAI

In [ ]:
client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "openai/gpt-oss-120b"

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a friendly bookstore assistant."},
        {"role": "user", "content": "Recommend a good book."}
    ],
)

print(response.choices[0].message.content)

In [ ]:
SYSTEM = """ROLE: You are the assistant for ReadMore Online Bookstore.

FACTS: We sell fiction, non-fiction, academic, and children's books.
       Orders depend on stock availability.
       Standard and express shipping are available.

RULES: Recommend books based on customer interests.
       Never invent book prices or availability.
       Ask questions if book details are unclear.
       Never guarantee delivery dates.
       If you don't know, say "I'll check with the store."
       Provide accurate order and return information.

FORMAT: Reply politely in under 60 words.
        Use friendly English or Telugu matching the customer."""

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    temperature=0.2,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": "Can you recommend a book for students?"}
    ],
)

print(response.choices[0].message.content)
print(response.usage)

In [ ]:
MENU = {
    "the alchemist": 350,
    "atomic habits": 450,
    "rich dad poor dad": 300,
    "harry potter": 550,
    "wings of fire": 250
}

MENU["atomic habits"]

MENU.get("donut", "Book not available")

In [ ]:
order = {
    "name": "Ravi",
    "item": "the alchemist",
    "qty": 2
}

order["qty"] * MENU[order["item"]]

In [ ]:
import json

text = json.dumps(order)

back = json.loads(text)

text

In [ ]:
back

In [ ]:
def get_price(item: str, qty: int = 1) -> int:
    return MENU[item.lower().strip()] * qty

get_price("atomic habits", 2)
get_price("The Alchemist")

In [ ]:
import functools

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"[log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper


@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for selected books."""
    return MENU[item.lower().strip()] * qty


get_price("atomic habits", 2)
get_price("The Alchemist")

In [ ]:
from openai import RateLimitError, APIConnectionError

history = [{"role": "system", "content": SYSTEM}]

MAX_TURNS = 8

def chat(user_text: str) -> str:

    history.append({"role": "user", "content": user_text})

    recent = [history[0]] + history[1:][-MAX_TURNS:]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )

        reply = response.choices[0].message.content

    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."

    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."

    history.append({"role": "assistant", "content": reply})

    return reply

In [ ]:
while True:

    text = input("You: ")

    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting ReadMore Online Bookstore!")
        break

    print("Bot:", chat(text))

In [ ]:
history